In [1]:
# 파이썬의 모든 자료형은 인스턴스
# iterable은 반복 돌릴 수 있는 인스턴스 
# 데이터 작업을 위한 기본 요소 - torch.utils.data, torch.utils.data.Dataset
# Dataset은 샘플과 정답(label)을 저장
# 여기서 샘플은 입력 데이터를 뜻하고 정답은 말 그대로 사람이 지정한 정답
# DataLoader는 Dataset을 순회 가능한 객체(iterable)로 감싼다.

import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import ToTensor

In [2]:
# 공개 데이터셋에서 학습 데이터를 내려받기
training_data = datasets.FashionMNIST(
    root="data",
    train=True,
    download=True,
    transform=ToTensor(),
)

# 공개 데이터셋에서 테스트 데이터를 내려받기
test_data = datasets.FashionMNIST(
    root="data",
    train=False,
    download=True,
    transform=ToTensor(),
)

In [3]:
batch_size = 64

# 데이터로더 생성
train_dataloader = DataLoader(training_data, batch_size=batch_size)
test_dataloader = DataLoader(test_data, batch_size=batch_size)

# test_dataloader에서 하나씩 꺼낸 값을 구조분해할당
for X, y in test_dataloader:
  print(f"Shape of X [N, C, H, W]: {X.shape}")
  print(f"Shape of y: {y.shape} {y.dtype}")
  break

Shape of X [N, C, H, W]: torch.Size([64, 1, 28, 28])
Shape of y: torch.Size([64]) torch.int64


In [4]:
# 학습에 사용할 CPU나 GPU, MPS 장치를 얻기
# 삼항 중첩
device = (
	"cuda"
	if torch.cuda.is_available()
	else "mps"
	if torch.backends.mps.is_available()
	else "cpu"
)

print(f"Using {device} device")

# 모델 정의
class NeuralNetwork(nn.Module):
  def __init__(self):
    super().__init__()
    self.flatten = nn.Flatten()
    self.linear_relu_stack = nn.Sequential(
      nn.Linear(28*28, 512),
      nn.ReLU(),
      nn.Linear(512, 512),
      nn.ReLU(),
      nn.Linear(512, 10)
    )

  def forward(self, x):
    x = self.flatten(x)
    logits = self.linear_relu_stack(x)
    return logits
  
model = NeuralNetwork().to(device)
print(model)

Using cpu device
NeuralNetwork(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (linear_relu_stack): Sequential(
    (0): Linear(in_features=784, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=512, bias=True)
    (3): ReLU()
    (4): Linear(in_features=512, out_features=10, bias=True)
  )
)


In [5]:
# 모델학습을 위한 손실함수와 옵티마이저 생성
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr = 1e-3)

In [6]:
def train(dataloader, model, loss_fn, optimizer):
  size = len(dataloader.dataset)
  for batch, (X, y) in enumerate(dataloader):
    X, y = X.to(device), y.to(device)

    # 예측 오류 계산
    pred = model(X)
    loss = loss_fn(pred, y)

    # 역전파
    loss.backward()
    optimizer.step()
    optimizer.zero_grad()

    if batch % 100 == 0:
      loss, current = loss.item(), (batch + 1) * len(X)
      print(f"loss: {loss:>7f}  [{current:>5d}/{size:>5d}]")

In [7]:
def test(dataloader, model, loss_fn):
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    model.eval()
    test_loss, correct = 0, 0
    with torch.no_grad():
        for X, y in dataloader:
            X, y = X.to(device), y.to(device)
            pred = model(X)
            test_loss += loss_fn(pred, y).item()
            correct += (pred.argmax(1) == y).type(torch.float).sum().item()
    test_loss /= num_batches
    correct /= size
    print(f"Test Error: \n Accuracy: {(100*correct):>0.1f}%, Avg loss: {test_loss:>8f} \n")

In [8]:
epochs = 5
for t in range(epochs):
    print(f"Epoch {t+1}\n-------------------------------")
    train(train_dataloader, model, loss_fn, optimizer)
    test(test_dataloader, model, loss_fn)
print("Done!")


Epoch 1
-------------------------------
loss: 2.305293  [   64/60000]
loss: 2.286276  [ 6464/60000]
loss: 2.264425  [12864/60000]
loss: 2.254393  [19264/60000]
loss: 2.234285  [25664/60000]
loss: 2.205569  [32064/60000]
loss: 2.214379  [38464/60000]
loss: 2.178569  [44864/60000]
loss: 2.183870  [51264/60000]
loss: 2.139096  [57664/60000]
Test Error: 
 Accuracy: 40.2%, Avg loss: 2.132950 

Epoch 2
-------------------------------
loss: 2.153248  [   64/60000]
loss: 2.133497  [ 6464/60000]
loss: 2.065291  [12864/60000]
loss: 2.082856  [19264/60000]
loss: 2.025408  [25664/60000]
loss: 1.964947  [32064/60000]
loss: 1.997413  [38464/60000]
loss: 1.911007  [44864/60000]
loss: 1.927825  [51264/60000]
loss: 1.843438  [57664/60000]
Test Error: 
 Accuracy: 52.6%, Avg loss: 1.841311 

Epoch 3
-------------------------------
loss: 1.883422  [   64/60000]
loss: 1.845046  [ 6464/60000]
loss: 1.717266  [12864/60000]
loss: 1.764729  [19264/60000]
loss: 1.652074  [25664/60000]
loss: 1.608954  [32064/600